# BEV-WAM-X Stage 3.1
Select a Colab GPU runtime with an NVIDIA A100. Change the two Drive paths in the setup cell, then run the notebook. The run directory holds verified checkpoints and logs so training can resume after a runtime reset.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json
import os
import subprocess
import sys

PACK = Path('/content/drive/MyDrive/bev-wam-x/stage31b-v2-pack')
RUN = Path('/content/drive/MyDrive/bev-wam-x/stage31b-v2-run')
REPO = Path('/content/bev-wam-x')
BRANCH = 'feat/v3-stage3-prep'
BATCH = 8
ACCUMULATE = 2
NUM_WORKERS = 4
MAX_UPDATES = 10000
SCHEDULE_UPDATES = 10000
WARMUP_UPDATES = 500
AUTO_UNASSIGN = True

def run(*args, cwd=None):
    subprocess.run(list(map(str, args)), cwd=cwd, check=True)

if not REPO.exists():
    run('git', 'clone', '--single-branch', '--branch', BRANCH,
        'https://github.com/AuroNoxx/bev-wam-x.git', REPO)
else:
    run('git', 'pull', '--ff-only', 'origin', BRANCH, cwd=REPO)
run('git', 'rev-parse', 'HEAD', cwd=REPO)
run(sys.executable, '-m', 'pip', 'install', '-r',
    REPO / 'v3_world_model/requirements_v3.txt')
import torch
assert torch.cuda.is_available() and not torch.version.hip
assert 'A100' in torch.cuda.get_device_name(0) and torch.cuda.is_bf16_supported(), torch.cuda.get_device_name(0)


In [ ]:
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from v3_world_model.pack_colab_v3 import read_manifest, sha256

source_manifest = PACK / 'manifest.json'
manifest = read_manifest(source_manifest, check_shards=False)
dataset = Path('/content') / ('stage31-' + sha256(source_manifest)[:16])
if not (dataset / 'manifest.json').is_file():
    run(sys.executable, '-m', 'v3_world_model.pack_colab_v3', '--unpack', source_manifest, '--out', dataset, cwd=REPO)
assert sha256(dataset / 'manifest.json') == sha256(source_manifest)
print(f"Verified extracted dataset with {manifest['total_frames']:,} frames")


In [ ]:
flags = ['--dataset', dataset, '--out', RUN, '--batch', BATCH, '--accumulate', ACCUMULATE,
         '--workers', NUM_WORKERS, '--max-updates', MAX_UPDATES, '--schedule-updates', SCHEDULE_UPDATES,
         '--warmup-updates', WARMUP_UPDATES, '--sdpa', 'cuda_flash', '--precision', 'bf16-mixed']
if (RUN / 'latest.ckpt').is_file():
    flags += ['--resume']
run(sys.executable, '-m', 'v3_world_model.train_stage31_colab', *flags, cwd=REPO)

result = json.loads((RUN / 'training-result.json').read_text(encoding='utf-8'))
assert result['actual_updates'] >= MAX_UPDATES and result['physical_batch'] == BATCH
assert result['gradient_accumulation'] == ACCUMULATE and result['clip_len'] == 4
assert result['pack_sha256'] == sha256(source_manifest)
for name in ('best.ckpt', 'latest.ckpt'):
    stored = torch.load(RUN / name, map_location='cpu', weights_only=False)
    assert stored['task_state']['pack_sha256'] == result['pack_sha256']
    assert stored['state_dict'] and stored['optimizer_states'] and stored['lr_schedulers']
    assert 0 < stored['global_step'] <= result['actual_updates']
    if name == 'latest.ckpt':
        assert stored['global_step'] == result['actual_updates']
    del stored
print(json.dumps(result, indent=2))
if AUTO_UNASSIGN:
    from google.colab import runtime
    runtime.unassign()
